In [1]:
!pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

In [31]:
import onnxruntime as ort
import librosa
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm


In [4]:

onnx_path = '/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_v2.onnx'
session_option = ort.SessionOptions()
session_option.intra_op_num_threads = 4
onnx_session = ort.InferenceSession(str(onnx_path), session_options = session_option, providers = ['CPUExecutionProvider'])
onnx_ipt_name = onnx_session.get_inputs()[0].name
# print(f'Onnx input ame: {onnx_ipt_name}')
onnx_opt_map = {o.name: i for i, o in enumerate(onnx_session.get_outputs())}
# print(f'Onnx output maps: {onnx_opt_map}')

## Load a 5-second .ogg chunk → numpy array at 32 kHz

Perch v2 expects `float32` waveform, shape `(batch, 160000)`, at **32 kHz**.

In [5]:

SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: float) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
# sample_path = '/kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1114648.ogg'  
# chunk = load_chunk(sample_path, offset_sec=0.0)
# print('Shape:', chunk.shape)   # (160000,)
# print('dtype:', chunk.dtype)   # float32
# print('Range:', chunk.min(), chunk.max())

In [6]:
def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = waveform[np.newaxis, :]
    outs = onnx_session.run(None, {onnx_ipt_name: inp})
    emb  = outs[onnx_opt_map['embedding']].astype(np.float32)
    return emb  # (1536,)

# emb = extract_embedding(chunk)
# print(f'Embedding shape: {emb.shape}')

In [7]:
NUM_CLASSES = 234
EMBED_DIM = 1536

class PerchHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(EMBED_DIM),
            nn.Dropout(0.3),
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x)

In [8]:
head = PerchHead()
use_previous_model = False
if use_previous_model:
    head_path = '/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_v2_cpu_onnx_head.pth'
    head.load_state_dict(torch.load(head_path, weights_only = False, map_location = 'cpu'))
# head.to('cpu')


In [27]:
def train():
    chunks_df = pd.read_parquet('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/chunks.parquet')
    BATCH_SIZE  = 512
    NUM_EPOCHS  = 10
    LR          = 1e-3
    N         = len(chunks_df) # number of chunks
    EMB_CACHE = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_embeddings.npy')
    LBL_CACHE = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_labels.npy')
    
    embeddings = np.memmap(EMB_CACHE, dtype='float32', mode='r', shape=(N, EMBED_DIM))
    labels     = np.memmap(LBL_CACHE, dtype='int64',   mode='r', shape=(N,))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Device: ', device)
    head.to(device)

    # ---- Dataset from cached numpy arrays ----
    X = torch.from_numpy(embeddings)   # (N, 1536) float32
    y = torch.from_numpy(labels)       # (N,)      int64
    
    dataset = TensorDataset(X, y)
    n_train = int(0.8 * len(dataset))
    n_val   = int(0.1 * len(dataset))
    n_test  = len(dataset) - n_train - n_val
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                              generator=torch.Generator().manual_seed(42))
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    base_dir = Path('/kaggle/working/')
    previous_run_num = len([p for p in base_dir.iterdir() if p.is_dir() and p.name.isdigit()])
    run_dir = base_dir / str(previous_run_num+1)
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f'Saving models to: {run_dir}')
    
    best_val_loss = float('inf')
        
    for epoch in tqdm(range(NUM_EPOCHS), desc=f'Training {NUM_EPOCHS} epochs ...'):
        # --- train ---
        head.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- validate ---
        head.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = head(xb)
                val_loss += criterion(preds, yb).item()
                correct  += (preds.argmax(1) == yb).sum().item()
                total    += len(yb)

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        val_acc     = correct / total
        scheduler.step()
        
        if (epoch+1)%50 == 0:
            print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS}  '
                f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(head.state_dict(), run_dir / 'best_model.pth')

    torch.save(head.state_dict(), run_dir / 'last_model.pth')
    print(f'Training complete. Best val loss: {best_val_loss:.4f}')

    

In [11]:
def submit():

    # Class labels from taxonomy.csv
    class_labels = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')['primary_label'].tolist()
    
    # List of test soundscapes (only visible during submission)
    test_soundscape_path = '/kaggle/input/competitions/birdclef-2026/test_soundscapes/'
    test_soundscapes = [os.path.join(test_soundscape_path, afile) for afile in sorted(os.listdir(test_soundscape_path)) if afile.endswith('.ogg')]
    
    # Open each soundscape and make predictions for 5-second segments
    # Use pandas df with 'row_id' plus class labels as columns
    predictions = pd.DataFrame(columns=['row_id'] + class_labels)
    for soundscape in test_soundscapes:
        print(soundscape)
        # Load audio
        sig, rate = librosa.load(path=soundscape, sr=32000)
    
        # Split into 5-second chunks
        chunks = []
        for i in range(0, len(sig), rate*5):
            chunk = sig[i:i+rate*5]
            chunks.append(chunk)
            
        # Make predictions for each chunk
        head.eval()
        with torch.no_grad():
            for i, chunk in enumerate(chunks):
                
                # Get row id  (soundscape id + end time of 5s chunk)      
                row_id = os.path.basename(soundscape).split('.')[0] + f'_{i * 5 + 5}'
        
                if len(chunk) < rate * 5:
                    chunk = np.pad(chunk, (0, rate * 5 - len(chunk)))
                
                # Transform the waveform into a melspectrogram to be able to feed in to the pretrained imagenet_b0 model
                # waveform = torch.tensor(chunk).unsqueeze(0)  # (1, samples)
                emb = extract_embedding(chunk)
                preds = head(torch.from_numpy(emb))
                probs = torch.softmax(preds, dim=1).cpu().numpy()[0]
                
                # Append to predictions as new row
                new_row = pd.DataFrame([[row_id] + list(probs)], columns=['row_id'] + class_labels)
                predictions = pd.concat([predictions, new_row], axis=0, ignore_index=True)
            
    # Save prediction as csv
    predictions.to_csv('submission.csv', index=False)
    #predictions.head()

In [32]:
MODE = 'train'
if MODE == 'train':
    train()
elif MODE == 'submit':
    submit()

Saving models to: /kaggle/working/2


Training 10 epochs ...:   0%|          | 0/10 [00:00<?, ?it/s]

Training complete. Best val loss: 1.5807
